# 04 — LLM Classification

Classify verbs into EO operators using OpenAI or Anthropic APIs.
Supports batch classification and single-verb testing.

**Requires:** API key set in 00_Setup.

In [ ]:
# ═══ Operator definitions for the classification prompt ═══
import json, re
from pyodide.http import pyfetch
from collections import Counter

OPERATOR_PROMPT = """You are classifying verb meanings into nine transformation types.
For each verb, determine which single operator best describes the
transformation the verb enacts.

THE NINE OPERATORS:
NUL — MARK ABSENCE: Something was present; now it's gone.
DES — DRAW DISTINCTION: Register something as different from its ground.
INS — SOMETHING APPEARS: Create a new event, entity, or state.
SEG — ONE BECOMES MANY: Split a unity into parts.
CON — CREATE PERSISTENT LINK: Establish a lasting relationship.
SYN — MANY BECOME ONE: Combine elements into a new unified whole.
ALT — CHANGE STATE: Same entity, different state.
SUP — HOLD INCOMPATIBLE: Multiple contradictory values coexist.
REC — REBUILD AROUND NEW CENTER: Reorganize structure around a different principle.

RULES:
- Choose the SINGLE best operator
- Note confidence (high/medium/low) and alternative if uncertain
- Consider the verb's most common usage"""

HELIX = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']

try:
    OPENAI_API_KEY
    ANTHROPIC_API_KEY
except NameError:
    OPENAI_API_KEY = ''
    ANTHROPIC_API_KEY = ''

print('Classification prompt loaded.')
print(f'OpenAI key: {"set" if OPENAI_API_KEY else "not set"}')
print(f'Anthropic key: {"set" if ANTHROPIC_API_KEY else "not set"}')

In [ ]:
# ═══ Classify a single verb ═══
VERB_TO_CLASSIFY = 'metamorphose'  # Change this!

prompt = f"""Classify this verb: {VERB_TO_CLASSIFY}

Respond in JSON format:
{{"verb": "{VERB_TO_CLASSIFY}", "operator": "...", "confidence": "...", "gloss": "...", "alternative": "", "reason": "..."}}"""

if ANTHROPIC_API_KEY:
    result = await anthropic_classify(prompt, OPERATOR_PROMPT)
    print(f'Response:\n{result}')
elif OPENAI_API_KEY:
    result = await api_call(
        'https://api.openai.com/v1/chat/completions',
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {OPENAI_API_KEY}'},
        {'model': 'gpt-4o-mini', 'messages': [
            {'role': 'system', 'content': OPERATOR_PROMPT},
            {'role': 'user', 'content': prompt}
        ], 'temperature': 0.3}
    )
    print(f'Response:\n{result["choices"][0]["message"]["content"]}')
else:
    print('No API key set. Configure in 00_Setup notebook.')

In [ ]:
# ═══ Batch classify verbs ═══
# Load corpus
try:
    corpus
except NameError:
    resp = await pyfetch('../data/combined_corpus.json')
    corpus = json.loads(await resp.string())

# Take a sample for classification
SAMPLE_SIZE = 100  # Change this (costs ~$0.01-0.10 depending on LLM)
verbs_to_classify = list(corpus.keys())[:SAMPLE_SIZE]

print(f'Will classify {len(verbs_to_classify)} verbs')
print(f'Sample: {verbs_to_classify[:10]}')

BATCH_SIZE = 50
all_classifications = []

for batch_start in range(0, len(verbs_to_classify), BATCH_SIZE):
    batch = verbs_to_classify[batch_start:batch_start + BATCH_SIZE]
    verb_list = '\n'.join(f'  {v}' for v in batch)
    
    prompt = f"""Classify these verbs into EO operators.
For each verb, provide: verb, operator, confidence, gloss, alternative.

VERBS:
{verb_list}

Respond in JSON array format:
[{{"verb": "...", "operator": "...", "confidence": "...", "gloss": "...", "alternative": ""}}]"""

    print(f'  Batch {batch_start//BATCH_SIZE + 1}: classifying {len(batch)} verbs...')
    
    try:
        if ANTHROPIC_API_KEY:
            text = await anthropic_classify(prompt, OPERATOR_PROMPT)
        elif OPENAI_API_KEY:
            result = await api_call(
                'https://api.openai.com/v1/chat/completions',
                {'Content-Type': 'application/json', 'Authorization': f'Bearer {OPENAI_API_KEY}'},
                {'model': 'gpt-4o-mini', 'messages': [
                    {'role': 'system', 'content': OPERATOR_PROMPT},
                    {'role': 'user', 'content': prompt}
                ], 'temperature': 0.3, 'max_tokens': 4096}
            )
            text = result['choices'][0]['message']['content']
        else:
            print('No API key set!')
            break
        
        # Extract JSON
        match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
        if match: text = match.group(1).strip()
        batch_results = json.loads(text)
        all_classifications.extend(batch_results)
        print(f'    Got {len(batch_results)} classifications')
    except Exception as e:
        print(f'    Error: {e}')
    
    # Rate limiting
    import asyncio
    await asyncio.sleep(1)

print(f'\nTotal classified: {len(all_classifications)}')

In [ ]:
# ═══ View classification results ═══
if all_classifications:
    # Distribution
    dist = Counter(c.get('operator', '?') for c in all_classifications)
    conf_dist = Counter(c.get('confidence', '?') for c in all_classifications)
    
    print(f'Operator distribution ({len(all_classifications)} verbs):')
    for op in HELIX:
        count = dist.get(op, 0)
        pct = count / len(all_classifications) * 100
        bar = '█' * int(pct * 2)
        print(f'  {op:>5s} {count:>5d} ({pct:>5.1f}%)  {bar}')
    
    print(f'\nConfidence: {dict(conf_dist)}')
    
    print(f'\nSample classifications:')
    print(f'{"Verb":>20s} {"Op":>5s} {"Conf":>6s} {"Gloss"}')
    print('-' * 65)
    for c in all_classifications[:30]:
        print(f'{c.get("verb",""):>20s} {c.get("operator",""):>5s} {c.get("confidence",""):>6s} {c.get("gloss","")[:35]}')
else:
    # Try loading pre-computed
    try:
        resp = await pyfetch('../data/llm_classifications.json')
        cls_data = json.loads(await resp.string())
        all_classifications = cls_data.get('classifications', [])
        print(f'Loaded {len(all_classifications)} pre-computed classifications')
        
        dist = Counter(c.get('operator', '?') for c in all_classifications)
        for op in HELIX:
            count = dist.get(op, 0)
            pct = count / len(all_classifications) * 100
            print(f'  {op:>5s} {count:>5d} ({pct:>5.1f}%)')
    except:
        print('No classifications available. Run the batch classification cell above.')

In [ ]:
# ═══ Download classifications ═══
if all_classifications:
    # download_json({'classifications': all_classifications}, 'eo_classifications.json')
    print(f'{len(all_classifications)} classifications ready for download.')
    print('Uncomment download_json() above to save.')